# FunctionGemma 270M Fine-Tuning — SU Lab Dataset

**Model**: google/functiongemma-270m-it
**Dataset**: eyalnof123/su-lab-functiongemma-dataset (1,303 train / 145 eval)
**Method**: Unsloth + LoRA (r=32, alpha=64)
**Output**: eyalnof123/functiongemma-270m-su-lab + GGUF Q4_K_M

Run all cells top to bottom. ~15-20 min on T4.

In [ ]:
# Step 1: Install dependencies
!pip install -q unsloth trl datasets peft accelerate bitsandbytes huggingface_hub

In [ ]:
# Step 2: Load model
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="google/functiongemma-270m-it",
    max_seq_length=4096,
    load_in_4bit=False,
)
print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {sum(p.numel() for p in model.parameters()) / 1e6:.1f}M")

In [ ]:
# Step 3: Apply LoRA
model = FastLanguageModel.get_peft_model(
    model,
    r=32,
    lora_alpha=64,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({trainable/total*100:.2f}%)")

In [ ]:
# Step 4: Load dataset from HuggingFace
from datasets import load_dataset

dataset = load_dataset("eyalnof123/su-lab-functiongemma-dataset")
train_dataset = dataset["train"]
print(f"Training examples: {len(train_dataset)}")
print(f"Sample: {train_dataset[0]['text'][:200]}...")

In [ ]:
# Step 5: Train
import os
os.environ["WANDB_DISABLED"] = "true"

from trl import SFTTrainer, SFTConfig

training_args = SFTConfig(
    output_dir="./output",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    weight_decay=0.01,
    lr_scheduler_type="linear",
    warmup_steps=5,
    logging_steps=10,
    save_strategy="epoch",
    bf16=True,
    fp16=False,
    optim="adamw_8bit",
    max_seq_length=4096,
    dataset_text_field="text",
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    tokenizer=tokenizer,
)

print("Starting training...")
trainer.train()
print("Training complete!")

In [ ]:
# Step 6: Save locally
model.save_pretrained("./output/final")
tokenizer.save_pretrained("./output/final")
print("Saved to ./output/final")

In [ ]:
# Step 7: Push to HuggingFace Hub
from huggingface_hub import login, HfApi, create_repo

# Login — paste your token when prompted
login()

REPO_ID = "eyalnof123/functiongemma-270m-su-lab"

try:
    create_repo(REPO_ID, private=False)
except:
    pass

api = HfApi()
api.upload_folder(
    folder_path="./output/final",
    repo_id=REPO_ID,
    repo_type="model",
)
print(f"Model pushed to https://huggingface.co/{REPO_ID}")

In [ ]:
# Step 8: Export GGUF Q4_K_M
print("Exporting GGUF Q4_K_M...")
model.save_pretrained_gguf("./output/gguf-q4", tokenizer, quantization_method="q4_k_m")

api.upload_folder(
    folder_path="./output/gguf-q4",
    repo_id=REPO_ID,
    repo_type="model",
    path_in_repo="gguf-q4",
)
print(f"GGUF Q4_K_M uploaded to {REPO_ID}/gguf-q4")

In [ ]:
# Step 9: Quick test
import re

FastLanguageModel.for_inference(model)

test_cases = [
    "make it red",
    "move left 50 pixels",
    "flashlight on",
    "make it spin",
    "add a button",
    "how are you",  # negative — should NOT call a function
]

for query in test_cases:
    prompt = f"<start_of_turn>user\n{query}\n<end_of_turn>\n<start_of_turn>model\n"
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    outputs = model.generate(**inputs, max_new_tokens=100, temperature=0.1, do_sample=True)
    result = tokenizer.decode(outputs[0], skip_special_tokens=False)
    # Extract model response
    model_part = result.split("<start_of_turn>model\n")[-1].split("<end_of_turn>")[0].strip()
    # Check for function call
    fc = re.search(r'<start_function_call>call:(\w+)\{(.*?)\}<end_function_call>', model_part)
    if fc:
        print(f"  {query:30s} -> {fc.group(1)}({fc.group(2)})")
    else:
        print(f"  {query:30s} -> (text) {model_part[:60]}")

print("\nDone! Model is ready.")
print(f"Download GGUF from: https://huggingface.co/{REPO_ID}/tree/main/gguf-q4")